# Spam Mail Detection System: Exploratory Data Analysis & Modeling

This notebook investigates the email spam dataset, performs text cleaning, explores word distributions, and evaluates a Multinomial Naive Bayes classifier using a Scikit-Learn Pipeline.

## 1. Imports and Environment Setup

In [ ]:
import re
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 5)

## 2. Load and Inspect Dataset

In [ ]:
data_path = Path("../data/spam_ham_dataset.csv")
if not data_path.exists():
    data_path = Path("data/spam_ham_dataset.csv")
df = pd.read_csv(data_path)
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# Inspect duplicates and nulls
print("Missing values:
", df.isnull().sum())
print("
Duplicate text messages:", df["text"].duplicated().sum())
df = df.drop_duplicates(subset=["text"]).dropna(subset=["text"]).reset_index(drop=True)
print("Shape after deduplication:", df.shape)

## 3. Class Distribution Analysis

In [ ]:
class_counts = df["label"].value_counts()
print("Class distribution:
", class_counts)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, ax=ax1, palette=["#3b82f6", "#ef4444"])
ax1.set_title("Message Count by Class")
ax1.set_ylabel("Count")

ax2.pie(class_counts.values, labels=class_counts.index, autopct="%1.1f%%", colors=["#3b82f6", "#ef4444"], startangle=140)
ax2.set_title("Class Proportion")
plt.tight_layout()
plt.show()

## 4. Message Length Analysis

In [ ]:
df["char_length"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=df, x="word_count", hue="label", kde=True, bins=50, ax=ax1, palette=["#3b82f6", "#ef4444"], log_scale=True)
ax1.set_title("Word Count Distribution (Log Scale)")

sns.boxplot(data=df, x="label", y="word_count", ax=ax2, palette=["#3b82f6", "#ef4444"])
ax2.set_title("Word Count Boxplot by Class")
ax2.set_ylim(0, 1500)
plt.tight_layout()
plt.show()

## 5. Text Cleaning and Preprocessing

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"^(subject|re|fwd):\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"https?://\S+|www\.\S+", " httpaddr ", text)
    text = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", " emailaddr ", text)
    text = re.sub(r"[$€£¥₹]", " currencysymb ", text)
    text = re.sub(r"\d+", " numtoken ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["cleaned_text"] = df["text"].apply(clean_text)
df[["text", "cleaned_text", "label"]].head()

## 6. Model Training & Evaluation via Scikit-Learn Pipeline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label_num"],
    test_size=0.2,
    random_state=42,
    stratify=df["label_num"],
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=clean_text, ngram_range=(1, 2), max_features=10000, sublinear_tf=True)),
    ("classifier", MultinomialNB(alpha=0.1)),
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("Test Accuracy: {:.2f}%".format(accuracy_score(y_test, y_pred) * 100))
print("Spam Precision: {:.2f}%".format(precision_score(y_test, y_pred) * 100))
print("Spam Recall:    {:.2f}%".format(recall_score(y_test, y_pred) * 100))
print("Spam F1-Score:  {:.2f}%".format(f1_score(y_test, y_pred) * 100))
print("
Classification Report:
", classification_report(y_test, y_pred, target_names=["Ham (0)", "Spam (1)"]))

## 7. Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Ham", "Spam"], yticklabels=["Ham", "Spam"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Held-Out Confusion Matrix")
plt.tight_layout()
plt.show()